In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import f1_score, accuracy_score
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# 1. CARGA DE DATOS ÓPTIMOS (10 variables del Wrapper de RandomForest)
# =============================================================================
train_df = pd.read_csv('train_selected_RandomForest.csv')
tuning_df = pd.read_csv('tuning_selected_RandomForest.csv')

target_col = 'Class'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_tuning = tuning_df.drop(columns=[target_col])
y_tuning = tuning_df[target_col]

# Codificamos el target a números
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_tuning_num = le.transform(y_tuning)

# =============================================================================
# 2. CONFIGURACIÓN DEL PREDEFINED SPLIT (Garantía de no Data Leakage)
# =============================================================================
# Combinamos los conjuntos para que el optimizador los procese juntos
X_combined = pd.concat([X_train, X_tuning], axis=0).reset_index(drop=True)
y_combined = np.concatenate([y_train_num, y_tuning_num])

# Creamos un índice: -1 significa "usar para entrenar", 0 significa "usar para validar/tuning"
split_index = [-1] * len(X_train) + [0] * len(X_tuning)
pds = PredefinedSplit(test_fold=split_index)

# Identificamos si es binario o multiclase para la métrica Juez (F1-Score)
is_multiclass = len(le.classes_) > 2
scoring_metric = 'f1_macro' if is_multiclass else 'f1'

print(f"Modo de optimización iniciado. Métrica objetivo: {scoring_metric}\n")

# =============================================================================
# 3. ESPACIOS DE BÚSQUEDA (Grids de Hiperparámetros)
# =============================================================================
param_dist_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

param_dist_svm = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

# Espacio de búsqueda para Regresión Logística
param_dist_lr = {
    'C': [0.01, 0.1, 1, 10, 100],          # Inverso de la fuerza de regularización
    'penalty': ['l1', 'l2'],               # Tipo de penalización
    'solver': ['liblinear', 'saga']        # Algoritmos que soportan penalización L1 y L2
}

# Espacio de búsqueda para k-NN
param_dist_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],   # Número de vecinos a consultar
    'weights': ['uniform', 'distance'],    # Peso de los vecinos (igual para todos o por cercanía)
    'p': [1, 2]                            # 1 = Distancia Manhattan, 2 = Distancia Euclidiana
}

# =============================================================================
# 4. EJECUCIÓN DE RANDOM SEARCH
# =============================================================================
# --- OPTIMIZACIÓN RANDOM FOREST ---
print("Ejecutando Random Search para Random Forest...")
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=15, 
    cv=pds,    
    scoring=scoring_metric,
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_combined, y_combined)

# --- OPTIMIZACIÓN SVM ---
print("Ejecutando Random Search para SVM...")
svm_search = RandomizedSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_distributions=param_dist_svm,
    n_iter=15, 
    cv=pds,    
    scoring=scoring_metric,
    random_state=42,
    n_jobs=-1
)
svm_search.fit(X_combined, y_combined)

# --- OPTIMIZACIÓN REGRESIÓN LOGÍSTICA ---
print("Ejecutando Random Search para Regresión Logística...")
lr_search = RandomizedSearchCV(
    estimator=LogisticRegression(random_state=42, max_iter=1000), # max_iter alto para asegurar convergencia
    param_distributions=param_dist_lr,
    n_iter=15, 
    cv=pds,    
    scoring=scoring_metric,
    random_state=42,
    n_jobs=-1
)
lr_search.fit(X_combined, y_combined)

# --- OPTIMIZACIÓN k-NN ---
print("Ejecutando Random Search para k-NN...")
knn_search = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=param_dist_knn,
    n_iter=15, 
    cv=pds,    
    scoring=scoring_metric,
    random_state=42,
    n_jobs=-1
)
knn_search.fit(X_combined, y_combined)

# =============================================================================
# 5. MOSTRAR RESULTADOS SELECCIONADOS
# =============================================================================
print("\n" + "="*60)
print("HIPERPARÁMETROS SELECCIONADOS (GANADORES)")
print("="*60)

print(f"▶ RANDOM FOREST:")
print(f"   Mejores Parámetros: {rf_search.best_params_}")
print(f"   F1-Score en Tuning : {rf_search.best_score_:.4f}\n")

print(f"▶ SVM:")
print(f"   Mejores Parámetros: {svm_search.best_params_}")
print(f"   F1-Score en Tuning : {svm_search.best_score_:.4f}\n")

print(f"▶ REGRESIÓN LOGÍSTICA:")
print(f"   Mejores Parámetros: {lr_search.best_params_}")
print(f"   F1-Score en Tuning : {lr_search.best_score_:.4f}\n")

print(f"▶ k-NN:")
print(f"   Mejores Parámetros: {knn_search.best_params_}")
print(f"   F1-Score en Tuning : {knn_search.best_score_:.4f}")
print("="*60)

ModuleNotFoundError: No module named 'pandas'